# Import

In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchbnn as bnn

device = "cuda" if torch.cuda.is_available() else "cpu"


# Read data

In [44]:
df = pd.read_csv('../data/data.csv')
df

,"Sigma, Mpa",T.K,t.h,Th,C,Cr,Co,Mo,W,Al,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni
0,241.316495,1144.2600,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,NaN,99.500
1,241.316495,1144.2600,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.75,NaN,NaN,NaN,99.250
2,241.316495,1144.2600,68.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.05,99.450
3,241.316495,1144.2600,40.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.20,99.300
4,241.316495,1144.2600,32.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.50,99.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,922.0389,1183.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3442,1103.161120,866.4833,25554.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3443,241.316495,1199.8170,183.2,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750
3444,241.316495,1199.8170,153.0,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750


In [45]:
mass = '232,03806	12,011	51,9961	58,933194	95,95	183,84	26,9815385	47,867	92,90637	10,81	55,845	88,90584	91,224	180,94788	186,207	101,07	50,9415	140,116	138,90547	32,06	28,085	54,938044	24,305	30,973761998	178,49	107,8682	63,546	208,9804	207,2	192,22	72,63	69,723	58,6934'.replace(',', '.').split()
element = 'Th	C	Cr	Co	Mo	W	Al	Ti	Nb	B	Fe	Y	Zr	Ta	Re	Ru	V	Ce	La	S	Si	Mn	Mg	P	Hf	Ag	Cu	Bi	Pb	Ir	Ge	Ga	Ni'.split()
atom_mass = dict(zip(element, mass))

In [46]:
atom_mass['Ni']

'58.6934'

In [47]:
for elem in df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list():
    df[elem] = df[elem] / float(atom_mass[elem])
df['sum'] = df[df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list()].sum(axis=1, skipna=True)
df['PLM'] = df['T.K'] * (20 + np.log10(df['t.h'])) * 1e-5

cols = df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns
df.loc[:, cols] = df.loc[:, cols].div(df['sum'], axis=0)
df.loc[:, cols] = df.loc[:, cols].div(df['Ni'], axis=0)



df = df.fillna(0)

df = df.drop(columns=['T.K', 't.h', 'Ni', 'sum'])

df

,"Sigma, Mpa",Th,C,Cr,Co,Mo,W,Al,Ti,Nb,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,PLM
0,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001652,0.000000,0.0,0.000000,0.139405
1,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.002485,0.000000,0.0,0.000000,0.149239
2,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001653,0.000000,0.0,0.000423,0.147465
3,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001656,0.000000,0.0,0.001695,0.146154
4,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001661,0.000000,0.0,0.004252,0.145925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,0.112115,0.020497,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.368295
3442,1103.161120,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,0.112115,0.020497,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.366118
3443,241.316495,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,0.112870,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.262391
3444,241.316495,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,0.112870,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.261469


In [48]:
target_col = 'Sigma, Mpa'

y = df[target_col]
X = df.copy().drop(columns=[target_col])


In [49]:
y_values = df[target_col].values.reshape(-1, 1).astype("float32")
X_values = X.values.astype("float32")

feature_names = list(X.columns)

print("X shape:", X_values.shape)
print("Number of input features:", X_values.shape[1])
print("feature_names:", feature_names)
df

X shape: (3446, 27)
Number of input features: 27
feature_names: ['Th', 'C', 'Cr', 'Co', 'Mo', 'W', 'Al', 'Ti', 'Nb', 'B', 'Fe', 'Y', 'Zr', 'Ta', 'Re', 'Ru', 'V', 'La', 'S', 'Si', 'Mn', 'P', 'Hf', 'Cu', 'Ge', 'Ga', 'PLM']


,"Sigma, Mpa",Th,C,Cr,Co,Mo,W,Al,Ti,Nb,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,PLM
0,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001652,0.000000,0.0,0.000000,0.139405
1,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.002485,0.000000,0.0,0.000000,0.149239
2,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001653,0.000000,0.0,0.000423,0.147465
3,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001656,0.000000,0.0,0.001695,0.146154
4,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001661,0.000000,0.0,0.004252,0.145925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,0.112115,0.020497,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.368295
3442,1103.161120,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,0.112115,0.020497,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.366118
3443,241.316495,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,0.112870,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.262391
3444,241.316495,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,0.112870,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.261469


In [50]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_temp, y_train, y_temp = train_test_split(
    X_values,
    y_values,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    shuffle=True
)


In [51]:
scale = True

if scale:
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train).astype("float32")
    X_val_scaled = scaler_X.transform(X_val).astype("float32")
    X_test_scaled = scaler_X.transform(X_test).astype("float32")

    y_train_scaled = scaler_y.fit_transform(y_train).astype("float32")
    y_val_scaled = scaler_y.transform(y_val).astype("float32")
    y_test_scaled = scaler_y.transform(y_test).astype("float32")
else:
    X_train_scaled = X_train
    X_val_scaled   = X_val
    X_test_scaled  = X_test
    y_train_scaled = y_train
    y_val_scaled   = y_val
    y_test_scaled  = y_test


In [52]:
class AlloyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [53]:
batch_size = 32

train_loader = DataLoader(
    AlloyDataset(X_train_scaled, y_train_scaled),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    AlloyDataset(X_val_scaled, y_val_scaled),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    AlloyDataset(X_test_scaled, y_test_scaled),
    batch_size=batch_size,
    shuffle=False
)


In [54]:
def train(
    model,
    train_loader,
    val_loader,
    lr=1e-3,
    weight_decay=1e-5,
    max_epochs=1000,
    patience=80,
    device="cpu"
):
    model = model.to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    history = {
        "train_loss": [],
        "val_loss": []
    }

    for epoch in tqdm(range(max_epochs), desc='training', leave=False):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            pred = model(xb)
            loss = criterion(pred, yb)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = criterion(pred, yb)

                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_state)

    return model, best_val_loss, history


In [55]:
def predict_scaled(model, loader, device="cpu"):
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)

            pred = model(xb).cpu().numpy()

            preds.append(pred)
            targets.append(yb.numpy())

    preds = np.vstack(preds)
    targets = np.vstack(targets)

    return preds, targets

def inverse_y(y_scaled, scaler_y):
    return scaler_y.inverse_transform(y_scaled)


In [56]:
def compute_metrics(y_pred, y_true):
    y_pred = y_pred.reshape(-1)
    y_true = y_true.reshape(-1)

    mae = np.mean(np.abs(y_pred - y_true))
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))

    rmae_percent = np.mean(np.abs((y_pred - y_true) / y_true)) * 100

    rrmse = rmse / np.mean(y_true)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "RMAE_percent": rmae_percent,
        "RRMSE": rrmse
    }


In [57]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

input_dim = X_train_scaled.shape[1]

results = []

for branch_hidden_dim in tqdm(range(4, 21), desc="Hidden dim search"):
    torch.manual_seed(42)
    np.random.seed(42)

    model = DifferentiatingSuperalloyNetV2(
        input_dim=input_dim,
        group_indices=group_indices,
        branch_hidden_dim=branch_hidden_dim,
        head_hidden_dim=32,
        dropout=0.0
    )

    model, best_val_loss, history = train_one_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=1e-3,
        weight_decay=1e-5,
        max_epochs=100,
        patience=80,
        device=device
    )

    y_val_pred_scaled, y_val_true_scaled = predict_scaled(
        model,
        val_loader,
        device=device
    )

    if scale:
        y_val_pred = inverse_y(y_val_pred_scaled, scaler_y)
        y_val_true = inverse_y(y_val_true_scaled, scaler_y)
    else:
        y_val_pred = y_val_pred_scaled
        y_val_true = y_val_true_scaled

    val_metrics = compute_metrics(y_val_pred, y_val_true)

    result = {
        "branch_hidden_dim": branch_hidden_dim,
        "best_val_loss_scaled": best_val_loss,
        **val_metrics
    }

    results.append(result)

    print(
        f"branch_hidden={branch_hidden_dim:2d} | "
        f"MAE={val_metrics['MAE']:.3f} | "
        f"RMSE={val_metrics['RMSE']:.3f} | "
        f"RMAE={val_metrics['RMAE_percent']:.2f}% | "
        f"RRMSE={val_metrics['RRMSE']:.4f}"
    )


Device: cpu


Hidden dim search:   0%|          | 0/17 [00:00<?, ?it/s]


NameError: name 'DifferentiatingSuperalloyNetV2' is not defined

In [ ]:
if scale:
        y_val_prey_train_origd = inverse_y(y_train_scaled, scaler_y)
        y_val_orig = inverse_y(y_val_scaled, scaler_y)
else:
        y_train_orig = y_train_scaled
        y_val_orig = y_val_scaled


baseline_pred = np.full_like(
    y_val_orig,
    fill_value=y_train_orig.mean()
)

baseline_metrics = compute_metrics(
    baseline_pred,
    y_val_orig
)

baseline_metrics


{'MAE': np.float32(848.4719),
 'RMSE': np.float32(860.38574),
 'RMAE_percent': np.float32(71.52434),
 'RRMSE': np.float32(0.72959673)}

In [ ]:
print("X_train_scaled finite:", np.isfinite(X_train_scaled).all())
print("X_val_scaled finite:", np.isfinite(X_val_scaled).all())
print("y_train_scaled finite:", np.isfinite(y_train_scaled).all())
print("y_val_scaled finite:", np.isfinite(y_val_scaled).all())

print("X_train_scaled mean:", X_train_scaled.mean())
print("X_train_scaled std:", X_train_scaled.std())

print("y_train_scaled mean:", y_train_scaled.mean())
print("y_train_scaled std:", y_train_scaled.std())


X_train_scaled finite: True
X_val_scaled finite: True
y_train_scaled finite: True
y_val_scaled finite: True
X_train_scaled mean: 1.3669106e-08
X_train_scaled std: 1.0
y_train_scaled mean: 3.7403643e-07
y_train_scaled std: 0.99999994


In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("RMAE_percent")

results_df

,branch_hidden_dim,best_val_loss_scaled,MAE,RMSE,RMAE_percent,RRMSE
6,10,0.263992,41.455273,65.852875,3.547632,0.055842
0,4,0.259229,41.736134,65.978119,3.557903,0.055949
10,14,0.270506,42.552547,65.694138,3.614195,0.055708
15,19,0.284284,43.558971,67.384628,3.766475,0.057141
5,9,0.270138,44.062588,66.857979,3.786541,0.056695
4,8,0.274312,44.595753,69.069435,3.829659,0.058570
12,16,0.298511,44.881321,69.975082,3.838791,0.059338
9,13,0.290366,44.653683,68.683167,3.859261,0.058242
7,11,0.291597,44.910679,69.202599,3.885734,0.058683
13,17,0.293949,45.702175,68.870621,3.889005,0.058401


In [ ]:
best_branch_hidden_dim = int(results_df.iloc[0]["branch_hidden_dim"])

print("Best branch_hidden_dim:", best_branch_hidden_dim)


Best branch_hidden_dim: 10


In [ ]:
torch.manual_seed(42)
np.random.seed(42)

best_model = DifferentiatingSuperalloyNetV2(
    input_dim=input_dim,
    group_indices=group_indices,
    branch_hidden_dim=best_branch_hidden_dim,
    head_hidden_dim=32,
    dropout=0.0
)

best_model, best_val_loss, best_history = train_one_model(
    model=best_model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=1e-3,
    weight_decay=1e-5,
    max_epochs=1000,
    patience=80,
    device=device
)


In [ ]:
y_test_pred_scaled, y_test_true_scaled = predict_scaled(
    best_model,
    test_loader,
    device=device
)

y_test_pred = inverse_y(y_test_pred_scaled, scaler_y)
y_test_true = inverse_y(y_test_true_scaled, scaler_y)

test_metrics = compute_metrics(y_test_pred, y_test_true)

test_metrics


{'MAE': np.float32(24.792288),
 'RMSE': np.float32(47.825905),
 'RMAE_percent': np.float32(2.1264176),
 'RRMSE': np.float32(0.040927317)}

In [ ]:
pred_df = pd.DataFrame({
    "y_true": y_test_true.reshape(-1),
    "y_pred": y_test_pred.reshape(-1),
    "abs_error": np.abs(y_test_pred.reshape(-1) - y_test_true.reshape(-1)),
    "rel_error_percent": np.abs(
        (y_test_pred.reshape(-1) - y_test_true.reshape(-1)) 
        / y_test_true.reshape(-1)
    ) * 100
})

pred_df.head()


,y_true,y_pred,abs_error,rel_error_percent
0,1023.150024,1036.707886,13.557861,1.325110
1,1473.150024,1467.412842,5.737183,0.389450
2,1073.150024,1077.624512,4.474487,0.416949
3,1183.150024,1183.984131,0.834106,0.070499
4,1033.150024,1037.280151,4.130127,0.399761


In [ ]:
def build_relationship_matrix(groups, feature_names):
    element_names = element

    R = np.zeros((len(groups), len(feature_names)), dtype=np.float32)

    group_ids = list(groups.keys())

    for i, group_id in enumerate(group_ids):
        for el in groups[group_id]:
            if el in element_names:
                j = element_names.index(el)
                R[i, j] = 1.0

    return R, group_ids, element_names

R, group_ids, element_names = build_relationship_matrix(
    GROUPS,
    feature_names
)

print(R.shape)


(16, 28)


In [ ]:
mae = np.mean(np.abs(y_test_pred - y_test_true))

rmse = np.sqrt(
    np.mean((y_test_pred - y_test_true) ** 2)
)

rmae = np.mean(
    np.abs(y_test_pred - y_test_true) / np.abs(y_test_true)
) * 100

rrmse = rmse / np.mean(np.abs(y_test_true))

print("Test MAE:", mae)
print("Test RMSE:", rmse)
print("Test RMAE, %:", rmae)
print("Test RRMSE:", rrmse)


Test MAE: 24.792288
Test RMSE: 47.825905
Test RMAE, %: 2.1264176
Test RRMSE: 0.040927317
